In [2]:
from dotenv import load_dotenv
from openai import OpenAI
import json

load_dotenv()
openai_client = OpenAI()

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)


In [4]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [5]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()


In [6]:

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [7]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)
    
    result_json = json.dumps(result, indent=2)

    return {
        'type': 'function_call_output',
        'call_id': call.call_id,
        'output': result_json,
    }
        

In [8]:
def agent_loop(instructions, question, model='gpt-5.4-mini') -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question},
    ]


    it = 1 

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False 

        response = openai_client.responses.create(
            model = model,
            input = messages,
            tools = [search_tool],
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True 
            elif item.type =='message':
                print('ASSISTANT:')
                print(item.content[0].text)
        
        it += 1 
        if has_function_calls == False:
            break

In [9]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [10]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [11]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [12]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [13]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [14]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [15]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [16]:
result.cost

CostInfo(input_cost=Decimal('0.0024405'), output_cost=Decimal('0.001593'), total_cost=Decimal('0.0040335'))

In [17]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run local install Ollama FAQ"}', call_id='call_8

In [18]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [ ]:
runner.run()

-> Response received


-> Response received
